# 문화누리 만족활동 기반 선호분석 — 결과 한눈에 보기

이 노트북은 **모델을 다시 학습하지 않고 현재 저장된 결과를 읽어 핵심만 보여줍니다.**

- 전체 재계산: `00_preference_pipeline_run_all.ipynb`
- 빠른 결과 확인: 현재 노트북을 `Run All`
- 접근성·가맹점·이동시간은 이 선호결과에 포함되지 않음

실제 CSV·HTML 산출물은 `data/processed/preference_analysis/` 아래에 생성되며 Git에는 포함되지 않습니다. 이 노트북은 저장된 산출물을 표와 그래프로 읽습니다.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, Markdown, display

def find_project_root():
    configured = os.environ.get('ORACLE_PROJECT_ROOT')
    starts = ([Path(configured).expanduser()] if configured else []) + [Path.cwd()]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / '.git').exists() and (candidate / 'data').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Oracle-Project 저장소를 찾지 못했습니다.')

ROOT = find_project_root()
PREF_DIR = ROOT / 'data/processed/preference_analysis'
MODEL_DIR = PREF_DIR / 'model'
SPATIAL_DIR = PREF_DIR / 'spatial'
SELECT_CATEGORY = '관광지'  # 도서·음악·영상·공연·미술·문화체험·관광지·스포츠관람·체육시설
POLICY_CATEGORIES = ['도서', '음악', '영상', '공연', '미술', '문화체험', '관광지', '스포츠관람', '체육시설']

font_candidates = [
    Path('/System/Library/Fonts/AppleSDGothicNeo.ttc'),
    Path('/System/Library/Fonts/Supplemental/AppleGothic.ttf'),
    Path('C:/Windows/Fonts/malgun.ttf'),
    Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'),
]
font_path = next((path for path in font_candidates if path.exists()), None)
if font_path is not None:
    font_manager.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams.get('font.family', ['sans-serif'])[0])

required = [
    MODEL_DIR / 'multinomial_model_metadata.json',
    MODEL_DIR / 'model_score_2024_2025.csv',
    MODEL_DIR / 'sex_age_middle_category_preference_2024.csv',
    SPATIAL_DIR / 'dong_middle_category_preference_demand_2024.csv',
    SPATIAL_DIR / 'gu_middle_category_preference_demand_2024.csv',
    SPATIAL_DIR / 'spatial_validation_summary_2024.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'결과파일이 없습니다. 먼저 전체 실행 노트북을 실행하세요: {missing}')
if SELECT_CATEGORY not in POLICY_CATEGORIES:
    raise ValueError(f'SELECT_CATEGORY는 다음 중 하나여야 합니다: {POLICY_CATEGORIES}')
print('프로젝트:', ROOT)
print('선택 분야:', SELECT_CATEGORY)

## 1. 사진에 보인 파일은 어디서 생성되는가

아래 표의 `실제 산출물`이 모델·공간 계산에서 생성되는 원본 결과입니다.

In [ ]:
provenance = pd.DataFrame([
    ['10_결과_모델성능.csv', 'model/model_score_2024_2025.csv', 'train_model.py', '2024·2025 모델과 가중 사전확률 baseline 평가'],
    ['10-1_결과_C와_변수구조_비교.csv', 'model/multinomial_tuning_combined_cv.csv', 'train_model.py → modeling.py', '순차검증+응답자 5-Fold Log Loss로 변수구조와 C 비교'],
    ['11_결과_성연령별_선호확률.csv', 'model/sex_age_middle_category_preference_2024.csv', 'train_model.py → modeling.py', '최종모델의 성별×연령별 정책 9개+기타 절대확률'],
    ['12_결과_100m격자_잠재수요.csv', 'spatial/grid_middle_category_preference_demand_2024.csv', 'build_spatial_outputs.py → spatial_demand.py', '격자 성연령 대상자×절대확률의 합'],
    ['13_결과_행정동_잠재수요.csv', 'spatial/dong_middle_category_preference_demand_2024.csv', 'spatial_demand.py', '100m 격자 잠재수요를 행정동코드로 합산'],
    ['14_결과_자치구_잠재수요.csv', 'spatial/gu_middle_category_preference_demand_2024.csv', 'spatial_demand.py', '100m 격자 잠재수요를 자치구로 합산'],
    ['14-1_결과_공간검증요약.csv', 'spatial/spatial_validation_summary_2024.csv', 'spatial_demand.py', '확률합·총량·격자→동→구 보존 검증'],
    ['15·16 HTML 지도', 'spatial/maps/*.html', 'build_spatial_outputs.py → interactive_maps.py', '저장된 격자·행정동 결과를 클릭 가능한 지도에 표시'],
], columns=['실행모음 바로가기', '실제 산출물', '생성 코드', '의미'])
display(provenance)

## 2. 최종 모델 선택과 성능

`selected=True`가 실제 채택된 변수구조와 C입니다. 모델 선택은 Distribution Match가 아니라 **검증 Log Loss**를 중심으로 수행했습니다. 2025년은 선택에 쓰지 않은 시간 외 평가자료입니다.

In [ ]:
metadata = json.loads((MODEL_DIR / 'multinomial_model_metadata.json').read_text(encoding='utf-8'))
tuning = pd.read_csv(MODEL_DIR / 'multinomial_tuning_combined_cv.csv', encoding='utf-8-sig')
scores = pd.read_csv(MODEL_DIR / 'model_score_2024_2025.csv', encoding='utf-8-sig')
category_validation = pd.read_csv(MODEL_DIR / 'model_validation_2025_by_category.csv', encoding='utf-8-sig')

display(Markdown('### 채택 설정'))
display(tuning.loc[tuning['selected'], [
    'feature_mode', 'c_value', 'temporal_mean_log_loss', 'grouped_mean_log_loss',
    'combined_mean_log_loss', 'combined_log_loss_score', 'selection_at_search_boundary'
]].round(4))
display(Markdown('### 2024·2025 모델과 baseline'))
display(scores[[
    'evaluation_year', 'model', 'accuracy', 'top3_accuracy', 'log_loss',
    'multiclass_brier', 'distribution_match_score', 'log_loss_skill_score_vs_baseline'
]].round(4))

category_view = category_validation.copy()
for column in ['observed_rate', 'predicted_rate']:
    category_view[column] = (category_view[column] * 100).round(2)
display(Markdown('### 2025 분야별 실제 가중비율과 평균 예측확률 (%)'))
display(category_view[[
    'middle_category', 'observed_rate', 'predicted_rate', 'absolute_difference_pp'
]].sort_values('observed_rate', ascending=False).reset_index(drop=True))

## 3. 성별×연령별 절대 선호확률

각 셀의 정책 9개 분야 확률 합은 기타확률 때문에 100%보다 작을 수 있습니다. 이는 정상이며, 절대 잠재수요에는 이 절대확률을 사용합니다.

In [ ]:
sex_age = pd.read_csv(MODEL_DIR / 'sex_age_middle_category_preference_2024.csv', encoding='utf-8-sig')
policy_probability = sex_age.loc[sex_age['is_policy_category']].copy()
policy_probability['확률(%)'] = policy_probability['preference_probability_absolute'] * 100
age_order = policy_probability.sort_values('age_code')['age_label'].drop_duplicates().tolist()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), constrained_layout=True)
for ax, sex_label in zip(axes, ['남성', '여성'], strict=True):
    pivot = (policy_probability.loc[policy_probability['sex_label'].eq(sex_label)]
             .pivot(index='age_label', columns='middle_category', values='확률(%)')
             .reindex(index=age_order, columns=POLICY_CATEGORIES))
    sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.1f', linewidths=.4,
                cbar_kws={'label': '절대 선호확률 (%)'})
    ax.set_title(f'{sex_label}: 연령별 만족활동 기반 절대 선호확률')
    ax.set_xlabel('')
    ax.set_ylabel('연령대')
plt.show()

## 4. 공간 잠재수요 결과

분야를 바꾸려면 첫 번째 코드 셀의 `SELECT_CATEGORY`를 수정한 뒤 이 셀부터 다시 실행하세요. 100m 전체 결과는 71MB이므로 이 노트북에서는 다시 읽지 않고 지도에서 확인합니다.

In [ ]:
dong = pd.read_csv(SPATIAL_DIR / 'dong_middle_category_preference_demand_2024.csv', encoding='utf-8-sig')
gu = pd.read_csv(SPATIAL_DIR / 'gu_middle_category_preference_demand_2024.csv', encoding='utf-8-sig')
spatial_metadata = json.loads((SPATIAL_DIR / 'spatial_preference_run_metadata_2024.json').read_text(encoding='utf-8'))

global_demand = (gu.groupby('middle_category', as_index=False)['potential_demand_absolute']
                 .sum().set_index('middle_category').reindex(POLICY_CATEGORIES).reset_index())
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=global_demand, x='potential_demand_absolute', y='middle_category', ax=ax, color='#E45756')
ax.set_title('서울시 분야별 만족활동 기반 잠재수요')
ax.set_xlabel('추정 잠재수요 인원')
ax.set_ylabel('')
plt.show()

selected_dong = dong.loc[dong['middle_category'].eq(SELECT_CATEGORY)].copy()
display(Markdown(f'### {SELECT_CATEGORY}: 잠재수요 상위 행정동'))
display(selected_dong.nlargest(15, 'potential_demand_absolute')[[
    '시군구', '행정동', 'target_population_est', 'preference_probability_absolute',
    'potential_demand_absolute', 'other_probability_absolute'
]].round(4).reset_index(drop=True))
display(Markdown(f'### {SELECT_CATEGORY}: 절대 선호확률 상위 행정동'))
display(selected_dong.nlargest(15, 'preference_probability_absolute')[[
    '시군구', '행정동', 'target_population_est', 'preference_probability_absolute',
    'potential_demand_absolute', 'preference_share_conditional_mnc'
]].round(4).reset_index(drop=True))

## 5. 공간 보존검증과 외적 타당성

외적 타당성은 예측 선호와 실제 카드 이용의 방향성 비교이며 모델 Accuracy나 오차율이 아닙니다. 카드 자료의 자치구 기준이 이용자 거주지인지 가맹점 소재지인지 확인되지 않았다는 제한도 함께 봐야 합니다.

In [ ]:
spatial_validation = pd.read_csv(SPATIAL_DIR / 'spatial_validation_summary_2024.csv', encoding='utf-8-sig')
external_summary = pd.read_csv(SPATIAL_DIR / 'external_validation_2024_summary.csv', encoding='utf-8-sig')
display(Markdown('### 공간 계산 보존검증'))
display(spatial_validation)
display(Markdown('### 2024 문화누리 이용실적 외적 타당성 요약'))
display(external_summary[['metric', 'value', 'interpretation', 'card_geography_basis']])

## 6. 상세 결과와 지도 열기

HTML은 미리 계산된 정적 결과이며 열 때 모델을 다시 실행하지 않습니다.

In [ ]:
links = [
    ('모델 성능 CSV', MODEL_DIR / 'model_score_2024_2025.csv'),
    ('성별×연령별 선호확률 CSV', MODEL_DIR / 'sex_age_middle_category_preference_2024.csv'),
    ('100m 격자 결과 CSV', SPATIAL_DIR / 'grid_middle_category_preference_demand_2024.csv'),
    ('행정동 결과 CSV', SPATIAL_DIR / 'dong_middle_category_preference_demand_2024.csv'),
    ('자치구 결과 CSV', SPATIAL_DIR / 'gu_middle_category_preference_demand_2024.csv'),
    ('100m 격자 지도', SPATIAL_DIR / 'maps/grid_preference_demand_2024.html'),
    ('행정동 지도', SPATIAL_DIR / 'maps/dong_preference_demand_2024.html'),
]
for label, path in links:
    display(FileLink(str(path), result_html_prefix=f'{label}: '))
print('✓ 저장된 결과 확인 완료 — 모델 재학습은 실행하지 않았습니다.')